# 14 - 1D CNN Sensor Classification (Solution)

### Step 1 README: Environment Setup
This cell configures resilient imports and verifies the local environment for CNN training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "edge_ai_course").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise ModuleNotFoundError(
        "Could not find src/edge_ai_course. Run from inside the 'day 5 sensor' folder or a child folder."
    )

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from edge_ai_course.signal_generator import SignalConfig, CLASS_ID_TO_NAME, generate_source_signals
from edge_ai_course.signal_dataset import build_sensor_dataset, save_dataset_artifacts, assert_no_source_leakage
from edge_ai_course.signal_features import extract_features, fft_components
from edge_ai_course.signal_models import train_classical_models, train_cnn_model, convert_to_tflite
from edge_ai_course.signal_visualization import plot_signal_and_fft, plot_feature_distribution, plot_confusion_matrix, plot_comparison_bars
from edge_ai_course.model_analysis import (
    build_comparison_table,
    write_day5_report,
    estimate_classical_pipeline_memory,
    estimate_cnn_pipeline_memory,
)

print("Environment ready for Notebook 14")
print(f"Project root: {PROJECT_ROOT}")
print(f"NumPy version: {np.__version__}")

### Step 2 README: Train 1D CNN
This cell creates leakage-safe splits, trains the 1D CNN, and plots core evaluation metrics.

In [ ]:
bundle = build_sensor_dataset(config=SignalConfig(), mode="quick", window_size=256, overlap_fraction=0.5)
assert_no_source_leakage(bundle)
save_dataset_artifacts(bundle)

cnn_results = train_cnn_model(
    x_train=bundle["x_train"],
    y_train=bundle["y_train"],
    x_val=bundle["x_val"],
    y_val=bundle["y_val"],
    x_test=bundle["x_test"],
    y_test=bundle["y_test"],
    epochs=12,
    batch_size=64,
)

metric_names = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
metric_values = [cnn_results["metrics"][name] for name in metric_names]
plt.figure(figsize=(7, 3.8))
plt.bar(metric_names, metric_values, color="#756bb1")
plt.ylim(0, 1.05)
plt.title("1D CNN Test Metrics")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print(cnn_results["metrics"])

### Step 3 README: Learning Curves and Confusion Matrix
This cell visualizes optimization history and class-level prediction performance on the test set.

In [ ]:
# Training curves
hist = cnn_results["history"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(hist["loss"], label="train")
axes[0].plot(hist["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(hist["accuracy"], label="train")
axes[1].plot(hist["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

plot_confusion_matrix(
    np.array(cnn_results["metrics"]["confusion_matrix"]),
    [CLASS_ID_TO_NAME[i] for i in sorted(CLASS_ID_TO_NAME)],
    "1D CNN confusion matrix",
)

### Step 4 README: TFLite Export and Size Check
This cell converts the trained model to TFLite (FP32/INT8 when available) and plots artifact sizes.

In [ ]:
# Optional TFLite conversion
x_train_norm = ((bundle["x_train"] - cnn_results["normalization"]["mean"]) / cnn_results["normalization"]["std"])[..., np.newaxis]
tflite_info = convert_to_tflite(cnn_results["model"], representative_data=x_train_norm)

size_rows = []
if "fp32_path" in tflite_info:
    size_rows.append(("tflite_fp32", int(tflite_info.get("fp32_size_bytes", "0"))))
if "int8_path" in tflite_info and "int8_size_bytes" in tflite_info:
    size_rows.append(("tflite_int8", int(tflite_info.get("int8_size_bytes", "0"))))

if size_rows:
    labels = [name for name, _ in size_rows]
    values_kb = [size / 1024.0 for _, size in size_rows]
    plt.figure(figsize=(6, 3.5))
    plt.bar(labels, values_kb, color="#31a354")
    plt.title("TFLite Model Size Comparison")
    plt.ylabel("Size [KB]")
    plt.tight_layout()
    plt.show()

if "int8_error" in tflite_info:
    print("INT8 conversion warning:", tflite_info["int8_error"])

tflite_info